# D6_03 블렌딩(보팅)과 스태킹

> **오늘 질문:** 성격이 다른 모델을 섞으면 최고 단일 모델보다 나아질까?

섞을 모델: 로지스틱 회귀(선형) + 랜덤포레스트(배깅) + LightGBM(부스팅)

| 표시 | 의미 |
|---|---|
| 📋 제공 | 코드가 들어 있습니다. 실행만 하세요 |
| ✍️ 입력 | 설명을 보고 직접 입력하세요 |
| 💬 해석 | 결과를 보고 답을 적어 보세요 |

## 0. 준비: 3일차 Data Mart 📋

In [ ]:
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import f1_score, precision_score, recall_score
from lightgbm import LGBMClassifier

df = pd.read_parquet("../../data/factory_table.parquet")
df["temp_diff"] = df["process_temp"] - df["air_temp"]
df["power"] = df["torque"] * df["rpm"]
df["strain"] = df["tool_wear"] * df["torque"]
df["grade_num"] = df["grade"].map({"L": 0, "M": 1, "H": 2})
features = ["air_temp", "process_temp", "rpm", "torque", "tool_wear",
            "vibration", "temp_diff", "power", "strain", "grade_num"]

cut = int(len(df) * 0.8)
X_train, X_test = df[features].iloc[:cut], df[features].iloc[cut:]
y_train, y_test = df["failure"].iloc[:cut], df["failure"].iloc[cut:]

## 1. 기본 모델 세 개 ✍️

로지스틱 회귀는 표준화가 필요하므로 Pipeline으로 묶습니다. 이름과 모델을 짝지은 목록으로 만들어 둡니다.

> 변수 이름: `base_models`

In [ ]:
# 힌트: [("이름", 모델), ...] 목록

## 2. 소프트 보팅 ✍️

세 모델의 '고장 확률'을 평균해 판정합니다.

> 변수 이름: `vote`

In [ ]:
# 힌트: VotingClassifier(목록, voting="soft")

## 3. 스태킹 ✍️

세 모델의 예측을 입력으로 메타러너(로지스틱 회귀)가 한 번 더 학습합니다. `cv=5`는 누수를 막기 위해 '안 본 조각의 예측(OOF)'으로 메타러너를 학습하게 합니다. (10초 정도)

> 변수 이름: `stack`

In [ ]:
# 힌트: StackingClassifier(목록, final_estimator=LogisticRegression(...), cv=5)

## 4. 단일 모델과 비교 📋

In [ ]:
models = {}
for name, model in base_models:
    models[name] = model.fit(X_train, y_train)
models["소프트 보팅"] = vote
models["스태킹"] = stack

rows = []
for name, model in models.items():
    p = model.predict(X_test)
    rows.append({"모델": name, "정밀도": precision_score(y_test, p), "재현율": recall_score(y_test, p), "F1": f1_score(y_test, p)})
pd.DataFrame(rows).sort_values("F1", ascending=False).round(3)

## 5. 메타러너는 누구를 믿었나 ✍️

메타러너(로지스틱 회귀)의 계수가 클수록 그 기본 모델의 예측을 더 믿었다는 뜻입니다.

In [ ]:
# 힌트: stack.final_estimator_.coef_

## 정리 💬

**Q1.** 보팅과 스태킹은 최고 단일 모델보다 나아졌나요? 정밀도와 재현율은 어떻게 바뀌었나요?

> 답:


**Q2.** 이 결과로 볼 때, 현장에 스태킹 모델을 배포하겠습니까?

> 답:
